# Dina Fitness AI Tagger LoRA Training

Run this notebook on a Google Colab GPU runtime. In VS Code, install the Google Colab extension, open this notebook, choose **Select Kernel**, then choose **Colab**.

Start with `Qwen/Qwen2.5-0.5B-Instruct` as a smoke test. After the notebook works, switch to `Qwen/Qwen2.5-1.5B-Instruct`.

In [ ]:
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
!pip install -U "transformers==4.53.1" "datasets==4.0.0" "accelerate==1.14.0" "peft==0.19.1" "bitsandbytes==0.50.0" sentencepiece "protobuf>=5.29.1,<6.0dev"

In [ ]:
import torch, transformers, datasets, peft, bitsandbytes
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("peft", peft.__version__)

## Mount Google Drive

Upload the server package into this Drive folder:

`MyDrive/dina-training/`

The notebook will auto-detect the newest `.tar.gz` archive in that folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/dina-training')
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

archives = sorted(DRIVE_DATA_DIR.glob('*.tar.gz'), key=lambda p: p.stat().st_mtime, reverse=True)
print('Drive data folder:', DRIVE_DATA_DIR)
print('Found archives:')
for archive in archives:
    print(' -', archive.name, round(archive.stat().st_size / 1024 / 1024, 2), 'MB')

if not archives:
    raise FileNotFoundError('No .tar.gz archive found. Upload dina-training-datasets-2026-08-04.tar.gz to MyDrive/dina-training first.')

preferred = [p for p in archives if p.name.startswith('dina-training-datasets')]
DATA_ARCHIVE = preferred[0] if preferred else archives[0]
WORK_DIR = Path('/content/dina-training')

# Smoke test first. Change to Qwen/Qwen2.5-1.5B-Instruct after the full flow works.
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = Path('/content/drive/MyDrive/dina-training/models/dina-tagger-qwen2.5-0.5b-lora')

print('Using archive:', DATA_ARCHIVE)
print('output:', OUTPUT_DIR)

In [ ]:
from pathlib import Path
import urllib.request

DATA_URL = "https://portal.fitnesswithdina.com/datasets/dina-training-datasets-2026-08-04.tar.gz"
DATA_ARCHIVE = Path("/content/dina-training-datasets-2026-08-04.tar.gz")
WORK_DIR = Path("/content/dina-training")

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = Path("/content/dina-models/dina-tagger-qwen2.5-0.5b-lora")

print("Downloading:", DATA_URL)
urllib.request.urlretrieve(DATA_URL, DATA_ARCHIVE)

print("archive exists:", DATA_ARCHIVE.exists(), DATA_ARCHIVE)
print("archive size MB:", round(DATA_ARCHIVE.stat().st_size / 1024 / 1024, 2))
print("output:", OUTPUT_DIR)

In [ ]:
import shutil
import tarfile

if not DATA_ARCHIVE.exists():
    raise FileNotFoundError(f'Archive not found: {DATA_ARCHIVE}')

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

with tarfile.open(DATA_ARCHIVE, 'r:gz') as archive:
    archive.extractall(WORK_DIR)

merged_dir = WORK_DIR / 'merged'
if not merged_dir.exists():
    raise FileNotFoundError(f'Expected merged dataset folder not found: {merged_dir}')

for path in sorted(merged_dir.glob('*.jsonl')):
    with path.open('r', encoding='utf-8') as f:
        rows = sum(1 for _ in f)
    print(rows, path)


## Install Training Libraries

In [ ]:
!pip install -U transformers datasets accelerate peft bitsandbytes sentencepiece protobuf

In [ ]:
!pip install -U "protobuf>=5.29.1,<6.0dev"

In [ ]:
import os
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = Path("/content/dina-models/dina-tagger-qwen2.5-0.5b-lora")

os.environ["BASE_MODEL"] = MODEL_ID
os.environ["OUT_DIR"] = str(OUTPUT_DIR)
os.environ["DATA_DIR"] = "/content/dina-training/merged"
os.environ["MAX_LENGTH"] = "2048"
os.environ["EPOCHS"] = "5"

!python /content/train_dina_tagger_lora.py

## Write Training Script

This trains only on `merged-strong-*`, which is the human-approved Dina dataset. Do not train first on wger/UCF101.

In [ ]:
%%writefile /content/train_dina_tagger_lora.py
import json
import os
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

if not torch.cuda.is_available():
    raise SystemExit('No CUDA GPU found. Change Colab runtime type to GPU.')

base = Path(os.environ.get('DATA_DIR', '/content/dina-training/merged'))
model_id = os.environ.get('BASE_MODEL', 'Qwen/Qwen2.5-0.5B-Instruct')
out_dir = os.environ.get('OUT_DIR', '/content/drive/MyDrive/dina-training/models/dina-tagger-qwen2.5-0.5b-lora')
max_length = int(os.environ.get('MAX_LENGTH', '2048'))
epochs = float(os.environ.get('EPOCHS', '5'))

print('base dataset:', base)
print('model:', model_id)
print('output:', out_dir)
print('max_length:', max_length)
print('epochs:', epochs)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_chat_jsonl(path):
    texts = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            messages = row.get('messages')
            if not messages:
                continue
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            texts.append(text)
    return Dataset.from_dict({'text': texts})

train_ds = load_chat_jsonl(base / 'merged-strong-train.jsonl')
eval_ds = load_chat_jsonl(base / 'merged-strong-validation.jsonl')
print('train rows:', len(train_ds), 'eval rows:', len(eval_ds))

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=max_length, padding=False)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'])
eval_ds = eval_ds.map(tokenize, batched=True, remove_columns=['text'])

bf16 = torch.cuda.is_bf16_supported()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if bf16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

training_kwargs = dict(
    output_dir=out_dir,
    num_train_epochs=epochs,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=5,
    save_strategy='epoch',
    eval_strategy='epoch',
    report_to='none',
    fp16=not bf16,
    bf16=bf16,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
)

try:
    args = TrainingArguments(**training_kwargs)
except TypeError:
    training_kwargs['evaluation_strategy'] = training_kwargs.pop('eval_strategy')
    args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()
trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)
print('Saved LoRA adapter to', out_dir)


## Start Training

For the first smoke test this uses 0.5B. After it succeeds, change `MODEL_ID` and `OUTPUT_DIR` above to 1.5B and rerun from the config cell.

In [ ]:
import os
from pathlib import Path

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = Path('/content/drive/MyDrive/dina-training/models/dina-tagger-qwen2.5-1.5b-lora')
os.environ["BASE_MODEL"] = MODEL_ID
os.environ["OUT_DIR"] = str(OUTPUT_DIR)
os.environ["DATA_DIR"] = "/content/dina-training/merged"
os.environ["MAX_LENGTH"] = "2048"
os.environ["EPOCHS"] = "5"

!python /content/train_dina_tagger_lora.py

## Quick Inference Test

In [ ]:
import json
import os
import subprocess
import textwrap
from pathlib import Path

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = Path('/content/drive/MyDrive/dina-training/models/dina-tagger-qwen2.5-1.5b-lora')

test_script = r'''
import json
import os
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = os.environ.get("BASE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
adapter_dir = Path(os.environ.get("OUT_DIR", "/content/dina-models/dina-tagger-qwen2.5-0.5b-lora"))

if not (adapter_dir / "adapter_config.json").exists():
    raise FileNotFoundError(f"Missing LoRA adapter: {adapter_dir}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Tokenizer must come from the base model. The LoRA folder only contains adapter weights.
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, str(adapter_dir))
model.eval()

messages = [
    {
        "role": "system",
        "content": "You are a fitness-library tagging expert for the Dina Fitness master exercise library. Return exactly one JSON object with keys tag, confidence, and reasoning. Inside tag, use only canonical Laravel field names: language, equipment_category, equipment_tags, primary_category, secondary_categories, training_adaptation, program_role, muscle_group, body_regions, exercise_type, exercise_family, movement_direction, stability_demand, variation_type, movement_patterns, training_styles, workout_sections, impact_level, intensity_level, video_variant, difficulty, confidence_bucket, recommended_repetitions, recommended_sets, recommended_rest_seconds. Never use aliases such as primary_tag, secondary_tags, program_section, stabilization_areas, or video_type.",
    },
    {
        "role": "user",
        "content": "Classify this exercise for Dina Fitness. Exercise metadata: {\"title\":\"Dumbbell Sumo Deadlift\", \"language\":\"en\", \"equipment\":\"dumbbells\", \"description\":\"Lower body hip hinge exercise for glutes and hamstrings.\"}. Return one raw JSON object only, with no markdown and no trailing semicolon. Approved values: equipment_category=[gym, full_gym, home_dumbbell, bodyweight]; primary_category=[resistance_training, cardiovascular_training, power_explosive_training, mobility, dynamic_warm_up, muscle_activation, flexibility_stretching, balance_stability, corrective_exercise, recovery_breathing, warm_up_cardio, cool_down_cardio, steady_state_cardio, optional_additional_cardio, hiit_cardio, post_workout_stretching, circuit_training]; training_adaptation=[general_fitness, strength, hypertrophy, muscular_endurance, power, explosiveness, speed, cardiovascular_endurance, anaerobic_conditioning, aerobic_conditioning, mobility, flexibility, stability, balance, coordination, muscle_activation, movement_preparation, rehabilitation_corrective, recovery]; program_role=[warm_up, warm_up_cardio, dynamic_warm_up, activation, lower_back_core_preparation, main_workout, main_compound_exercise, accessory_exercise, isolation_exercise, superset_exercise, circuit_exercise, hiit_interval, cardio, optional_cardio, finisher, core, cool_down, cool_down_stretching, post_workout_stretching, corrective, recovery]; body_regions=[full_body, upper_body, lower_body, chest, back, shoulders, arms, glutes, quadriceps, hamstrings, calves, core, abs, obliques, lower_back]; exercise_type=[resistance, main, bodyweight, dumbbell, gym, cardio, cardio_warm_up, warm_up, mobility, stretching, activation, power_explosive, lower_back, abs, obliques]; movement_patterns=[squat, hinge, lunge, push, pull, carry, rotation, anti_rotation, flexion, extension, abduction, adduction, stabilization, locomotion, jumping, crawling, mobility, stretching]; workout_sections=[dynamic_warm_up, warm_up_cardio, mobility_dynamic_warm_up, muscle_activation, lower_back_core_superset, core_lower_back_preparation, main_workout, core_obliques, lower_back_strengthening, optional_additional_cardio, post_workout_stretching, cool_down_stretching]; video_variant=[explained, no_audio]; difficulty=[beginner, intermediate, advanced]. For this exercise, prefer training_adaptation=hypertrophy or strength, body_regions=[lower_body, glutes, hamstrings], and exercise_type=resistance.",
    },
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=700,
        min_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_ids = output[0][inputs["input_ids"].shape[-1]:]
decoded = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
print("generated_token_count:", generated_ids.numel())
print("decoded_output:")
print(decoded if decoded else "<EMPTY>")
if not decoded:
    print("raw_decoded_repr:", repr(tokenizer.decode(generated_ids, skip_special_tokens=False)))

required_tag_keys = {
    "language", "equipment_category", "primary_category", "training_adaptation",
    "program_role", "muscle_group", "exercise_type", "difficulty",
}
alias_keys = {"primary_tag", "secondary_tags", "program_section", "stabilization_areas", "video_type"}
allowed_values = {
    "equipment_category": {"gym", "full_gym", "home_dumbbell", "bodyweight"},
    "primary_category": {"resistance_training", "cardiovascular_training", "power_explosive_training", "mobility", "dynamic_warm_up", "muscle_activation", "flexibility_stretching", "balance_stability", "corrective_exercise", "recovery_breathing", "warm_up_cardio", "cool_down_cardio", "steady_state_cardio", "optional_additional_cardio", "hiit_cardio", "post_workout_stretching", "circuit_training"},
    "training_adaptation": {"general_fitness", "strength", "hypertrophy", "muscular_endurance", "power", "explosiveness", "speed", "cardiovascular_endurance", "anaerobic_conditioning", "aerobic_conditioning", "mobility", "flexibility", "stability", "balance", "coordination", "muscle_activation", "movement_preparation", "rehabilitation_corrective", "recovery"},
    "program_role": {"warm_up", "warm_up_cardio", "dynamic_warm_up", "activation", "lower_back_core_preparation", "main_workout", "main_compound_exercise", "accessory_exercise", "isolation_exercise", "superset_exercise", "circuit_exercise", "hiit_interval", "cardio", "optional_cardio", "finisher", "core", "cool_down", "cool_down_stretching", "post_workout_stretching", "corrective", "recovery"},
    "body_regions": {"full_body", "upper_body", "lower_body", "chest", "back", "shoulders", "arms", "glutes", "quadriceps", "hamstrings", "calves", "core", "abs", "obliques", "lower_back"},
    "exercise_type": {"resistance", "main", "bodyweight", "dumbbell", "gym", "cardio", "cardio_warm_up", "warm_up", "mobility", "stretching", "activation", "power_explosive", "lower_back", "abs", "obliques"},
    "movement_patterns": {"squat", "hinge", "lunge", "push", "pull", "carry", "rotation", "anti_rotation", "flexion", "extension", "abduction", "adduction", "stabilization", "locomotion", "jumping", "crawling", "mobility", "stretching"},
    "workout_sections": {"dynamic_warm_up", "warm_up_cardio", "mobility_dynamic_warm_up", "muscle_activation", "lower_back_core_superset", "core_lower_back_preparation", "main_workout", "core_obliques", "lower_back_strengthening", "optional_additional_cardio", "post_workout_stretching", "cool_down_stretching"},
    "impact_level": {"low", "moderate", "high"},
    "intensity_level": {"low", "moderate", "high"},
    "video_variant": {"explained", "no_audio"},
    "difficulty": {"beginner", "intermediate", "advanced"},
    "confidence_bucket": {"high", "medium", "low"},
}

try:
    parsed, end = json.JSONDecoder().raw_decode(decoded)
    extra_text = decoded[end:].strip()
    tag = parsed.get("tag", {}) if isinstance(parsed, dict) else {}
    missing = sorted(required_tag_keys - set(tag.keys()))
    aliases = sorted(alias_keys & set(tag.keys()))
    invalid = []
    for field, allowed in allowed_values.items():
        values = tag.get(field, [])
        if values in (None, ""):
            continue
        if not isinstance(values, list):
            values = [values]
        invalid.extend(f"{field}={value}" for value in values if value not in allowed)
    print("validation:")
    if not missing and not aliases and not invalid and not extra_text:
        print("PASS")
    else:
        print("FAIL")
        print("missing_required_keys:", missing)
        print("alias_keys_present:", aliases)
        print("invalid_values:", invalid)
        print("extra_text_after_json:", repr(extra_text))

    repair_actions = []
    normalized_tag = dict(tag)
    scalar_aliases = {
        "training_adaptation": {"muscle_gain": "hypertrophy", "muscle_growth": "hypertrophy"},
        "exercise_type": {"lower_body": "resistance", "upper_body": "resistance", "strength": "resistance"},
    }
    array_aliases = {
        "body_regions": {"hips": None, "hip": None, "posterior_chain": "lower_body"},
        "movement_patterns": {"deadlift": "hinge"},
    }
    for field, aliases_for_field in scalar_aliases.items():
        value = normalized_tag.get(field)
        if value in aliases_for_field:
            normalized_tag[field] = aliases_for_field[value]
            repair_actions.append(f"{field}: {value} -> {normalized_tag[field]}")
    for field, aliases_for_field in array_aliases.items():
        values = normalized_tag.get(field, [])
        if not isinstance(values, list):
            values = [values]
        normalized_values = []
        for value in values:
            replacement = aliases_for_field.get(value, value)
            if replacement is None:
                repair_actions.append(f"{field}: dropped {value}")
                continue
            if replacement != value:
                repair_actions.append(f"{field}: {value} -> {replacement}")
            normalized_values.append(replacement)
        normalized_tag[field] = list(dict.fromkeys(normalized_values))
    normalized_payload = dict(parsed)
    normalized_payload["tag"] = normalized_tag
    normalized_missing = sorted(required_tag_keys - set(normalized_tag.keys()))
    normalized_aliases = sorted(alias_keys & set(normalized_tag.keys()))
    normalized_invalid = []
    for field, allowed in allowed_values.items():
        values = normalized_tag.get(field, [])
        if values in (None, ""):
            continue
        if not isinstance(values, list):
            values = [values]
        normalized_invalid.extend(f"{field}={value}" for value in values if value not in allowed)
    print("normalized_candidate:")
    print(json.dumps(normalized_payload, ensure_ascii=False, separators=(",", ":")))
    print("repair_actions:", repair_actions)
    print("normalized_validation:")
    if not normalized_missing and not normalized_aliases and not normalized_invalid:
        print("PASS")
    else:
        print("FAIL")
        print("missing_required_keys:", normalized_missing)
        print("alias_keys_present:", normalized_aliases)
        print("invalid_values:", normalized_invalid)
except Exception as exc:
    print("validation:")
    print("FAIL invalid_json", repr(exc))
'''

Path("/content/test_dina_tagger_lora.py").write_text(textwrap.dedent(test_script), encoding="utf-8")

env = os.environ.copy()
env["BASE_MODEL"] = MODEL_ID
env["OUT_DIR"] = str(OUTPUT_DIR)
result = subprocess.run(["python", "/content/test_dina_tagger_lora.py"], text=True, capture_output=True, env=env)
print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)
result.check_returncode()

## Batch Evaluation On Dina Exercises

Run this after the single Quick Inference Test. It evaluates the saved LoRA against exported Dina exercise examples from `merged-strong-test.jsonl`.

In [24]:
import os
import subprocess
import textwrap
from pathlib import Path

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = Path('/content/drive/MyDrive/dina-training/models/dina-tagger-qwen2.5-1.5b-lora')
DATA_FILE = Path('/content/dina-training/merged/merged-strong-test.jsonl')
EVAL_LIMIT = 30
RESULTS_FILE = Path('/content/dina-batch-eval-results.jsonl')
EXPECTED_EVALUATOR_VERSION = 'proposal_eval_v3'

batch_eval_script = r'''
import json
import os
import re
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = os.environ.get('BASE_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')
adapter_dir = Path(os.environ.get('OUT_DIR', '/content/drive/MyDrive/dina-training/models/dina-tagger-qwen2.5-1.5b-lora'))
data_file = Path(os.environ.get('DATA_FILE', '/content/dina-training/merged/merged-strong-test.jsonl'))
results_file = Path(os.environ.get('RESULTS_FILE', '/content/dina-batch-eval-results.jsonl'))
eval_limit = int(os.environ.get('EVAL_LIMIT', '30'))
evaluator_version = os.environ.get('EVALUATOR_VERSION', 'proposal_eval_v3')

if not data_file.exists():
    raise FileNotFoundError(f'Missing eval data file: {data_file}')
if not (adapter_dir / 'adapter_config.json').exists():
    raise FileNotFoundError(f'Missing LoRA adapter: {adapter_dir}')

SYSTEM_PROMPT = """You are a fitness-library tagging expert for the Dina Fitness master exercise library. Return one raw JSON object only, with no markdown and no trailing semicolon. Return exactly these top-level keys: tag, confidence, reasoning. Inside tag, use only canonical Laravel field names: language, equipment_category, equipment_tags, primary_category, secondary_categories, training_adaptation, program_role, muscle_group, secondary_muscle_groups, body_regions, exercise_type, exercise_family, movement_direction, stability_demand, variation_type, movement_patterns, training_styles, workout_sections, impact_level, intensity_level, video_variant, difficulty, confidence_bucket, recommended_repetitions, recommended_sets, recommended_rest_seconds. Never use aliases such as primary_tag, secondary_tags, program_section, stabilization_areas, or video_type. Use only approved values from the prompt and metadata."""

REQUIRED_TAG_KEYS = {'language', 'equipment_category', 'primary_category', 'training_adaptation', 'program_role', 'muscle_group', 'exercise_type', 'difficulty'}
REQUIRED_TOP_KEYS = {'tag', 'confidence', 'reasoning'}
ALIAS_KEYS = {'primary_tag', 'secondary_tags', 'program_section', 'stabilization_areas', 'video_type'}
ALLOWED_VALUES = {
    'equipment_category': {'gym', 'full_gym', 'home_dumbbell', 'bodyweight'},
    'primary_category': {'resistance_training', 'cardiovascular_training', 'power_explosive_training', 'mobility', 'dynamic_warm_up', 'muscle_activation', 'flexibility_stretching', 'balance_stability', 'corrective_exercise', 'recovery_breathing', 'warm_up_cardio', 'cool_down_cardio', 'steady_state_cardio', 'optional_additional_cardio', 'hiit_cardio', 'post_workout_stretching', 'circuit_training'},
    'training_adaptation': {'general_fitness', 'strength', 'hypertrophy', 'muscular_endurance', 'power', 'explosiveness', 'speed', 'cardiovascular_endurance', 'anaerobic_conditioning', 'aerobic_conditioning', 'mobility', 'flexibility', 'stability', 'balance', 'coordination', 'muscle_activation', 'movement_preparation', 'rehabilitation_corrective', 'recovery'},
    'program_role': {'warm_up', 'warm_up_cardio', 'dynamic_warm_up', 'activation', 'lower_back_core_preparation', 'main_workout', 'main_compound_exercise', 'accessory_exercise', 'isolation_exercise', 'superset_exercise', 'circuit_exercise', 'hiit_interval', 'cardio', 'optional_cardio', 'finisher', 'core', 'cool_down', 'cool_down_stretching', 'post_workout_stretching', 'corrective', 'recovery'},
    'body_regions': {'full_body', 'upper_body', 'lower_body', 'chest', 'back', 'shoulders', 'arms', 'glutes', 'quadriceps', 'hamstrings', 'calves', 'core', 'abs', 'obliques', 'lower_back'},
    'exercise_type': {'resistance', 'main', 'bodyweight', 'dumbbell', 'gym', 'cardio', 'cardio_warm_up', 'warm_up', 'mobility', 'stretching', 'activation', 'power_explosive', 'lower_back', 'abs', 'obliques'},
    'movement_patterns': {'squat', 'hinge', 'lunge', 'push', 'pull', 'carry', 'rotation', 'anti_rotation', 'flexion', 'extension', 'abduction', 'adduction', 'stabilization', 'locomotion', 'jumping', 'crawling', 'mobility', 'stretching'},
    'training_styles': {'resistance_training', 'hypertrophy', 'muscular_endurance', 'conditioning', 'mobility', 'core', 'stretching', 'warm_up', 'circuit', 'hiit', 'steady_state_cardio'},
    'workout_sections': {'dynamic_warm_up', 'warm_up_cardio', 'mobility_dynamic_warm_up', 'muscle_activation', 'lower_back_core_superset', 'core_lower_back_preparation', 'main_workout', 'core_obliques', 'lower_back_strengthening', 'optional_additional_cardio', 'post_workout_stretching', 'cool_down_stretching'},
    'impact_level': {'low', 'moderate', 'high'},
    'intensity_level': {'low', 'moderate', 'high'},
    'video_variant': {'explained', 'no_audio'},
    'difficulty': {'beginner', 'intermediate', 'advanced'},
    'confidence_bucket': {'high', 'medium', 'low'},
}
SCALAR_ALIASES = {
    'training_adaptation': {'muscle_gain': 'hypertrophy', 'muscle_growth': 'hypertrophy', 'core': 'stability', 'core_strength': 'strength', 'lower_back_strength': 'strength'},
    'exercise_type': {'lower': 'resistance', 'lower_body': 'resistance', 'upper_body': 'resistance', 'strength': 'resistance'},
    'program_role': {'abs': 'core', 'main_strength_weps_and_weights': 'main_workout'},
}
ARRAY_ALIASES = {
    'body_regions': {'hips': None, 'hip': None, 'middle_body': 'core', 'posterior_chain': 'lower_body'},
    'movement_patterns': {'deadlift': 'hinge'},
    'workout_sections': {'warm_up': 'dynamic_warm_up', 'post_workout': 'post_workout_stretching', 'main_core_superset': 'lower_back_core_superset', 'lower_back_superset': 'lower_back_core_superset', 'mobility': 'dynamic_warm_up'},
}
KEY_ALIASES = {
    'primary_tag': 'primary_category',
    'secondary_tags': 'secondary_categories',
    'program_section': 'program_role',
    'video_type': 'video_variant',
}

def load_examples(path, limit):
    rows = []
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            obj = json.loads(line)
            messages = obj.get('messages', [])
            user = next((m.get('content', '') for m in messages if m.get('role') == 'user'), '')
            expected_text = next((m.get('content', '') for m in messages if m.get('role') == 'assistant'), '{}')
            try:
                expected = json.loads(expected_text)
            except Exception:
                expected = {}
            rows.append({'user': user, 'expected': expected, 'title': title_from_user(user)})
            if len(rows) >= limit:
                break
    return rows

def title_from_user(user):
    match = re.search(r'"title"\s*:\s*"([^"]+)"', user)
    if match:
        return match.group(1)
    return user.splitlines()[0][:80] if user else 'unknown'

def escape_control_chars_in_strings(text):
    result = []
    in_string = False
    escaped = False
    for ch in text:
        if escaped:
            result.append(ch)
            escaped = False
            continue
        if ch == '\\':
            result.append(ch)
            escaped = True
            continue
        if ch == '"':
            result.append(ch)
            in_string = not in_string
            continue
        if in_string and ord(ch) < 32:
            if ch == '\n':
                result.append('\\n')
            elif ch == '\r':
                result.append('\\r')
            elif ch == '\t':
                result.append('\\t')
            else:
                result.append(' ')
            continue
        result.append(ch)
    return ''.join(result)

def extract_balanced_json_object(text):
    start = None
    depth = 0
    in_string = False
    escaped = False
    for i, ch in enumerate(text):
        if start is None:
            if ch == '{':
                start = i
                depth = 1
            continue
        if escaped:
            escaped = False
            continue
        if ch == '\\':
            escaped = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return text[start:i + 1], text[i + 1:].strip()
    return None, ''

def parse_json_variant(text):
    last_error = None
    for variant in (text, escape_control_chars_in_strings(text)):
        try:
            parsed, end = json.JSONDecoder().raw_decode(variant)
            return parsed, variant[end:].strip()
        except Exception as exc:
            last_error = exc
    raise last_error or ValueError('JSON parse failed')

def recover_tag_object(raw):
    match = re.search(r'"tag"\s*:', raw)
    if not match:
        return None
    brace_start = raw.find('{', match.end())
    if brace_start < 0:
        return None
    tag_text, _ = extract_balanced_json_object(raw[brace_start:])
    if not tag_text:
        return None
    try:
        tag, _ = parse_json_variant(tag_text)
    except Exception:
        return None
    if not isinstance(tag, dict):
        return None
    payload = {'tag': tag}
    confidence_match = re.search(r'"confidence"\s*:\s*(0(?:\.\d+)?|1(?:\.0+)?)', raw)
    if confidence_match:
        try:
            payload['confidence'] = float(confidence_match.group(1))
        except Exception:
            pass
    return payload

def parse_first_json(raw):
    start = raw.find('{')
    if start < 0:
        raise ValueError('no JSON object found')
    text = raw[start:].strip()
    balanced, balanced_extra = extract_balanced_json_object(text)
    candidates = []
    if balanced:
        candidates.append((balanced, balanced_extra))
    candidates.append((text, ''))
    last_error = None
    for candidate, forced_extra in candidates:
        try:
            parsed, extra = parse_json_variant(candidate)
            return parsed, extra or forced_extra
        except Exception as exc:
            last_error = exc
    recovered = recover_tag_object(raw)
    if recovered is not None:
        return recovered, 'recovered_tag_object_only'
    raise last_error or ValueError('JSON parse failed')

def validation_errors(payload, extra_text=''):
    errors = []
    if extra_text:
        errors.append(f'extra_text_after_json={extra_text!r}')
    if not isinstance(payload, dict):
        return ['payload_not_object']
    missing_top = sorted(REQUIRED_TOP_KEYS - set(payload.keys()))
    if missing_top:
        errors.append('missing_top_keys=' + ','.join(missing_top))
    tag = payload.get('tag', {})
    if not isinstance(tag, dict):
        return errors + ['tag_not_object']
    missing = sorted(REQUIRED_TAG_KEYS - set(tag.keys()))
    if missing:
        errors.append('missing_tag_keys=' + ','.join(missing))
    aliases = sorted(ALIAS_KEYS & set(tag.keys()))
    if aliases:
        errors.append('alias_keys=' + ','.join(aliases))
    if isinstance(tag.get('muscle_group'), list):
        errors.append('muscle_group_is_array')
    for field, allowed in ALLOWED_VALUES.items():
        values = tag.get(field, [])
        if values in (None, ''):
            continue
        if not isinstance(values, list):
            values = [values]
        for value in values:
            if value not in allowed:
                errors.append(f'invalid_{field}={value}')
    return errors

def normalize_payload(payload):
    payload = dict(payload) if isinstance(payload, dict) else {}
    if 'tag' not in payload:
        payload = {'tag': payload}
    tag = payload.get('tag', {})
    tag = dict(tag) if isinstance(tag, dict) else {}
    repair_actions = []
    if 'confidence' not in payload or payload.get('confidence') in (None, ''):
        bucket = payload.get('confidence_bucket') or tag.get('confidence_bucket')
        payload['confidence'] = {'high': 0.95, 'medium': 0.75, 'low': 0.45}.get(bucket, 0.6)
        repair_actions.append('missing confidence -> inferred from confidence_bucket/default')
    if 'reasoning' not in payload or not str(payload.get('reasoning', '')).strip():
        payload['reasoning'] = 'AI-generated proposal pending manual review.'
        repair_actions.append('missing reasoning -> default pending-review reasoning')
    for old_key, new_key in KEY_ALIASES.items():
        if old_key in tag and new_key not in tag:
            tag[new_key] = tag.pop(old_key)
            repair_actions.append(f'{old_key} -> {new_key}')
    muscle_group = tag.get('muscle_group')
    if isinstance(muscle_group, list):
        non_empty = [str(item).strip() for item in muscle_group if str(item).strip()]
        if non_empty:
            tag['muscle_group'] = non_empty[0]
            existing_secondary = tag.get('secondary_muscle_groups', [])
            if not isinstance(existing_secondary, list):
                existing_secondary = [existing_secondary]
            tag['secondary_muscle_groups'] = list(dict.fromkeys(non_empty[1:] + existing_secondary))
            repair_actions.append('muscle_group array -> primary + secondary_muscle_groups')
    for field, aliases in SCALAR_ALIASES.items():
        value = tag.get(field)
        if value in aliases:
            tag[field] = aliases[value]
            repair_actions.append(f'{field}: {value} -> {tag[field]}')
    for field, aliases in ARRAY_ALIASES.items():
        values = tag.get(field, [])
        if not isinstance(values, list):
            values = [values]
        normalized = []
        for value in values:
            replacement = aliases.get(value, value)
            if replacement is None:
                repair_actions.append(f'{field}: dropped {value}')
                continue
            if replacement != value:
                repair_actions.append(f'{field}: {value} -> {replacement}')
            normalized.append(replacement)
        tag[field] = list(dict.fromkeys(normalized))
    payload['tag'] = tag
    return payload, repair_actions

def generate_one(model, tokenizer, user_content):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=900, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

print('evaluator_version:', evaluator_version)
print('loading model:', model_id)
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, str(adapter_dir))
model.eval()

examples = load_examples(data_file, eval_limit)
summary = {'evaluator_version': evaluator_version, 'total': len(examples), 'raw_json_clean_pass': 0, 'raw_schema_pass': 0, 'normalized_schema_pass': 0, 'failed_after_normalization': 0}
failures = []
results_file.parent.mkdir(parents=True, exist_ok=True)
print('overwriting results_file:', results_file)
with results_file.open('w', encoding='utf-8') as out:
    for index, row in enumerate(examples, 1):
        print(f'[{index}/{len(examples)}] {row["title"]}', flush=True)
        raw = generate_one(model, tokenizer, row['user'])
        parsed = None
        extra = ''
        raw_errors = []
        normalized = None
        repair_actions = []
        normalized_errors = []
        try:
            parsed, extra = parse_first_json(raw)
            if not extra:
                summary['raw_json_clean_pass'] += 1
            raw_errors = validation_errors(parsed, extra)
            if not raw_errors:
                summary['raw_schema_pass'] += 1
            normalized, repair_actions = normalize_payload(parsed)
            normalized_errors = validation_errors(normalized, '')
            if not normalized_errors:
                summary['normalized_schema_pass'] += 1
            else:
                summary['failed_after_normalization'] += 1
        except Exception as exc:
            raw_errors = [f'invalid_json={exc!r}']
            normalized_errors = raw_errors
            summary['failed_after_normalization'] += 1
        record = {'evaluator_version': evaluator_version, 'index': index, 'title': row['title'], 'raw_errors': raw_errors, 'normalized_errors': normalized_errors, 'repair_actions': repair_actions, 'raw': raw, 'normalized': normalized}
        out.write(json.dumps(record, ensure_ascii=False) + '\n')
        if normalized_errors:
            failures.append(record)

print('\nsummary:')
print(json.dumps(summary, indent=2))
print('results_file:', results_file)
if failures:
    print('\nfirst_failures:')
    for failure in failures[:10]:
        print(f"- #{failure['index']} {failure['title']}: {failure['normalized_errors']}")
'''

Path('/content/evaluate_dina_tagger_lora.py').write_text(textwrap.dedent(batch_eval_script), encoding='utf-8')

env = os.environ.copy()
env['BASE_MODEL'] = MODEL_ID
env['OUT_DIR'] = str(OUTPUT_DIR)
env['DATA_FILE'] = str(DATA_FILE)
env['EVAL_LIMIT'] = str(EVAL_LIMIT)
env['RESULTS_FILE'] = str(RESULTS_FILE)
env['EVALUATOR_VERSION'] = 'proposal_eval_v3'
result = subprocess.run(['python', '/content/evaluate_dina_tagger_lora.py'], text=True, capture_output=True, env=env)
print(result.stdout)
if result.stderr:
    print('STDERR:\n', result.stderr)
result.check_returncode()

evaluator_version: proposal_eval_v3
loading model: Qwen/Qwen2.5-1.5B-Instruct
overwriting results_file: /content/dina-batch-eval-results.jsonl
[1/30] Sumo Deadlift
[2/30] Seated straight arm bicep curl
[3/30] 3 exercises/Bicep curl+front raise+tricep overhead
[4/30] Upright front shoulder raise with twist
[5/30] Single arm back row on bench
[6/30] Single arm back row on bench
[7/30] Incline chest press
[8/30] Hamstring Bridge
[9/30] Split squat back foot elevated
[10/30] Knee push up to feet
[11/30] EX 001A- Flat chest press
[12/30] Side plank hip taps
[13/30] Plank twists
[14/30] Core + oblique warm up
[15/30] 2 Glute & Hip activation exercises
[16/30] Lower body warm up
[17/30] Swing throughs + Jumping squats HIIT
[18/30] Seated rear row
[19/30] Split Single Leg Deadlift Smith Machine
[20/30] Step Up On Bench
[21/30] Assisted Pull up machine (wide grip)
[22/30] Kneeling Kick Back Machine
[23/30] Assisted Pull Up Machine
[24/30] Cable bicep curl
[25/30] Seated reverse fly machine
[26/

## Read Batch Evaluation Results

Run this after the batch cell if Colab truncates the output. It reads `/content/dina-batch-eval-results.jsonl` and prints the summary again.

In [25]:
from collections import Counter
from pathlib import Path
import json

RESULTS_FILE = Path('/content/dina-batch-eval-results.jsonl')
EXPECTED_EVALUATOR_VERSION = 'proposal_eval_v3'

print('results_file:', RESULTS_FILE)
print('exists:', RESULTS_FILE.exists())
if not RESULTS_FILE.exists():
    raise FileNotFoundError('Run Batch Evaluation On Dina Exercises first.')

rows = []
with RESULTS_FILE.open('r', encoding='utf-8') as handle:
    for line in handle:
        if line.strip():
            rows.append(json.loads(line))

versions = sorted({row.get('evaluator_version', 'missing') for row in rows})
print('evaluator_versions:', versions)
if rows and versions != [EXPECTED_EVALUATOR_VERSION]:
    raise RuntimeError(
        f'Stale result file. Expected {EXPECTED_EVALUATOR_VERSION}, got {versions}. '
        'Rerun the Batch Evaluation On Dina Exercises cell, then run this reader again.'
    )

raw_json_clean_pass = sum(
    1 for row in rows
    if not any(str(error).startswith('invalid_json=') or str(error).startswith('extra_text_after_json=') for error in row.get('raw_errors', []))
)
raw_schema_pass = sum(1 for row in rows if not row.get('raw_errors'))
normalized_schema_pass = sum(1 for row in rows if not row.get('normalized_errors'))
failed_after_normalization = sum(1 for row in rows if row.get('normalized_errors'))

summary = {
    'evaluator_version': EXPECTED_EVALUATOR_VERSION,
    'total': len(rows),
    'raw_json_clean_pass': raw_json_clean_pass,
    'raw_schema_pass': raw_schema_pass,
    'normalized_schema_pass': normalized_schema_pass,
    'failed_after_normalization': failed_after_normalization,
}
print('summary:')
print(json.dumps(summary, indent=2))

raw_error_counts = Counter(error for row in rows for error in row.get('raw_errors', []))
normalized_error_counts = Counter(error for row in rows for error in row.get('normalized_errors', []))
repair_counts = Counter(action for row in rows for action in row.get('repair_actions', []))

print('\ntop_raw_errors:')
for error, count in raw_error_counts.most_common(20):
    print(count, error)

print('\ntop_normalized_errors:')
for error, count in normalized_error_counts.most_common(20):
    print(count, error)

print('\ntop_repairs:')
for action, count in repair_counts.most_common(20):
    print(count, action)

print('\nfailed_rows:')
for row in rows:
    if row.get('normalized_errors'):
        print(f"#{row.get('index')} {row.get('title')}: {row.get('normalized_errors')}")

print('\nfirst_5_rows:')
for row in rows[:5]:
    print(f"#{row.get('index')} {row.get('title')} raw={row.get('raw_errors')} normalized={row.get('normalized_errors')}")


results_file: /content/dina-batch-eval-results.jsonl
exists: True
evaluator_versions: ['proposal_eval_v3']
summary:
{
  "evaluator_version": "proposal_eval_v3",
  "total": 30,
  "raw_json_clean_pass": 15,
  "raw_schema_pass": 0,
  "normalized_schema_pass": 25,
  "failed_after_normalization": 5
}

top_raw_errors:
19 missing_top_keys=confidence,reasoning
13 extra_text_after_json='recovered_tag_object_only'
10 missing_top_keys=reasoning
5 invalid_training_adaptation=core
4 invalid_workout_sections=warm_up
2 invalid_program_role=abs
2 invalid_workout_sections=lower_back_superset
2 invalid_workout_sections=post_workout
2 invalid_program_role=main_strength_weps_and_weights
1 extra_text_after_json='.'
1 invalid_training_adaptation=lower_back_strength
1 invalid_exercise_type=lower
1 invalid_training_styles=resistance
1 invalid_workout_sections=main_core_superset
1 invalid_workout_sections=mobility
1 invalid_workout_sections=lower_back_activation
1 invalid_workout_sections=main_strength_section

## Next Run: 1.5B

After the 0.5B smoke test succeeds, update:

```python
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = Path('/content/drive/MyDrive/dina-training/models/dina-tagger-qwen2.5-1.5b-lora')
```

Then rerun the training cells.